In [10]:
import sagemaker
import boto3
import pandas as pd
import numpy as np
import time

session = sagemaker.Session()

role = sagemaker.get_execution_role()
region = session.boto_region_name

bucket = "amazon-ml-challenge-aayushi"

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Region: us-east-1
Role: arn:aws:iam::191518191034:role/service-role/AmazonSageMaker-ExecutionRole-20260925T084535
Bucket: amazon-ml-challenge-aayushi


In [11]:
DATA_PREFIX = (
    "6ab10eb3b23ba_student_resource/"
    "student_resource/dataset"
)

TRAIN_PREFIX = f"{DATA_PREFIX}/train"
TEST_PREFIX = f"{DATA_PREFIX}/test"

print("Dataset:", DATA_PREFIX)
print("Train  :", TRAIN_PREFIX)
print("Test   :", TEST_PREFIX)

Dataset: 6ab10eb3b23ba_student_resource/student_resource/dataset
Train  : 6ab10eb3b23ba_student_resource/student_resource/dataset/train
Test   : 6ab10eb3b23ba_student_resource/student_resource/dataset/test


In [12]:
s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=DATA_PREFIX
)

for obj in response.get("Contents", []):
    print(obj["Key"])

6ab10eb3b23ba_student_resource/student_resource/dataset/.DS_Store
6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source1.tsv
6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source2.tsv
6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source3.tsv
6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_ground_truth.tsv
6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_source1.tsv
6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_source2.tsv
6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_source3.tsv


In [13]:
source1_s3 = f"s3://{bucket}/{TRAIN_PREFIX}/train_source1.tsv"

source1_sample = pd.read_csv(
    source1_s3,
    sep="\t",
    dtype="string",
    nrows=5
)

source1_sample

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [14]:
train_files = {
    "source1": f"s3://{bucket}/{TRAIN_PREFIX}/train_source1.tsv",
    "source2": f"s3://{bucket}/{TRAIN_PREFIX}/train_source2.tsv",
    "source3": f"s3://{bucket}/{TRAIN_PREFIX}/train_source3.tsv",
    "ground_truth": f"s3://{bucket}/{TRAIN_PREFIX}/train_ground_truth.tsv"
}

for name, path in train_files.items():
    print(f"\n===== {name} =====")
    
    sample = pd.read_csv(
        path,
        sep="\t",
        dtype="string",
        nrows=3
    )
    
    print(sample)
    print("Columns:", list(sample.columns))


===== source1 =====
      entity_id        business_name                        business_address  \
0  S1-925783039  Orelee's Barbershop  1795 Westchester Drive, High Point, NC   
1  S1-773889195          Prime Money         17560 Ellis Road, Tahlequah, OK   
2  S1-377745466        B+ Retail Inc     1712 Montebello Avenue, Phoenix, AZ   

  country  
0      US  
1      US  
2      US  
Columns: ['entity_id', 'business_name', 'business_address', 'country']

===== source2 =====
      entity_id                    business_name  \
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1  S2-764573417     -- Holloway Peak Inc Seafood   
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी   

                                   business_address country  
0      KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi   India  
1                         105 ELM ST, MORGANTON, NC      US  
2  G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh   India  
Columns: ['entity_id', 'business_name', 'business_address', '

In [15]:
for name, path in train_files.items():
    key = path.replace(f"s3://{bucket}/", "")
    
    obj = s3.head_object(
        Bucket=bucket,
        Key=key
    )
    
    size_gb = obj["ContentLength"] / (1024 ** 3)
    
    print(f"{name:15} {size_gb:.2f} GB")

source1         0.20 GB
source2         0.46 GB
source3         0.47 GB
ground_truth    0.12 GB


In [16]:
for name, path in train_files.items():
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype="string",
        chunksize=100_000
    ):
        total_rows += len(chunk)

    print(f"{name:15} {total_rows:,} rows")

source1         2,206,821 rows
source2         5,034,616 rows
source3         5,285,603 rows
ground_truth    2,206,821 rows


In [17]:
match_counts = []

gt_path = train_files["ground_truth"]

for chunk in pd.read_csv(
    gt_path,
    sep="\t",
    dtype="string",
    chunksize=100_000
):
    counts = (
        chunk["matched_entity_ids"]
        .fillna("")
        .apply(lambda x: len(x.split(",")) if x else 0)
    )
    
    match_counts.extend(counts.tolist())

match_counts = pd.Series(match_counts)

print("Total Source 1 entities:", len(match_counts))
print("Total positive matches:", match_counts.sum())
print("Average matches per Source 1:", match_counts.mean())
print("Maximum matches for one Source 1:", match_counts.max())

Total Source 1 entities: 2206821
Total positive matches: 7638365
Average matches per Source 1: 3.4612526344456573
Maximum matches for one Source 1: 11


In [20]:
import re
import unicodedata

def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    
    # Lowercase
    text = text.lower()
    
    # Replace &
    text = text.replace("&", " and ")
    
    # Remove punctuation but keep Unicode letters/marks/numbers
    text = "".join(
        char if (
            unicodedata.category(char).startswith(("L", "M", "N"))
            or char.isspace()
        ) else " "
        for char in text
    )
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [21]:
examples = [
    "Orelee's Barbershop",
    "B+ Retail Inc",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "LLC Moncada Léarning Center"
]

for x in examples:
    print("Original  :", x)
    print("Normalized:", normalize_text(x))
    print()

Original  : Orelee's Barbershop
Normalized: orelee s barbershop

Original  : B+ Retail Inc
Normalized: b retail inc

Original  : राम मार्केटिंग प्राइवेट लिमिटेड
Normalized: राम मार्केटिंग प्राइवेट लिमिटेड

Original  : LLC Moncada Léarning Center
Normalized: llc moncada léarning center



In [22]:
for name, path in train_files.items():
    if name == "ground_truth":
        continue
    
    print(f"\n===== {name} =====")
    
    sample = pd.read_csv(
        path,
        sep="\t",
        dtype="string",
        nrows=10000
    )
    
    print(sample.isna().sum())


===== source1 =====
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

===== source2 =====
entity_id             0
business_name         0
business_address    337
country               0
dtype: int64

===== source3 =====
entity_id             0
business_name         0
business_address    350
country               0
dtype: int64
